In [ ]:
# Cell #1: Load the model and print the state dictionary

import torch
from transformers import GPT2LMHeadModel

torch.manual_seed(3407)

model_hf = GPT2LMHeadModel.from_pretrained("gpt2", dtype=torch.float16)

for k, v in model_hf.state_dict().items():
    print(k, v.shape)

In [ ]:
# Cell #2: Build the tokenizer and run model inference

import tiktoken
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel

torch.manual_seed(3407)

model_hf = GPT2LMHeadModel.from_pretrained("gpt2", dtype=torch.float16)

device = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")
model_hf.to(device)
model_hf.eval()

num_return_sequences = 5
max_length = 16
top_k = 50

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode("Hello, I'm a language model,")
x = torch.tensor(tokens, dtype=torch.long, device=device)
x = x.unsqueeze(0).repeat(num_return_sequences, 1)  # (B, T)

with torch.no_grad():
    while x.size(1) < max_length:
        logits = model_hf(x).logits[:, -1, :]  # (B, vocab_size)
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, top_k, dim=-1)  # (B, top_k)
        ix = torch.multinomial(topk_probs, 1)  # (B, 1)
        next_token = torch.gather(topk_indices, -1, ix)  # (B, 1)
        x = torch.cat((x, next_token), dim=1)

for row in x.tolist():
    print(">", enc.decode(row))

In [ ]:
# Cell #3: Build the custom GPT2 model and run model inference

import tiktoken
import torch
import torch.nn.functional as F

from learn_gpt2.gpt2 import GPT2

model_custom = GPT2.from_pretrained("gpt2", dtype=torch.float16)

device = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")
model_custom.to(device)
model_custom.eval()

# Seed after loading so sampling matches Cell #2; random weight init consumes RNG.
torch.manual_seed(3407)

num_return_sequences = 5
max_length = 16
top_k = 50

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode("Hello, I'm a language model,")
x = torch.tensor(tokens, dtype=torch.long, device=device)
x = x.unsqueeze(0).repeat(num_return_sequences, 1)  # (B, T)

with torch.no_grad():
    while x.size(1) < max_length:
        logits = model_custom(x)[:, -1, :]  # (B, vocab_size)
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, top_k, dim=-1)  # (B, top_k)
        ix = torch.multinomial(topk_probs, 1)  # (B, 1)
        next_token = torch.gather(topk_indices, -1, ix)  # (B, 1)
        x = torch.cat((x, next_token), dim=1)

for row in x.tolist():
    print(">", enc.decode(row))